# SUNSHIELD v2 — Notebook 3: Anomaly Definition

**Input:** `step2_outputs` (Notebook 2). **Output:** `step3_outputs` (+ zip).

### The rule (one rule, applied to each line separately, for k = 1, 1.5, 2, 2.5, 3)

A reading is an anomaly when the inverter's **actual power deviates from the expected power** more than usual:

$$r_t = \text{actual}_t - \text{expected}_t \qquad\qquad \text{anomaly}_t = 1 \;\text{ if }\; r_t < \mu - k\sigma \;\text{ or }\; r_t > \mu + k\sigma$$

| | expected power | gives |
|---|---|---|
| **Actual line** (ground truth) | physical expected power, PVWatts (from irradiation + module temperature) | **Actual 0/1** |
| **Predicted line** (the model) | LSTM predicted power (trained on healthy readings) | **Predicted 0/1** |

- $\mu$ and $\sigma$ are computed **once, from the training days only**, for every inverter and for each line
  separately: $\mu$ = median of $r$, $\sigma = 1.4826 \cdot \text{MAD}(r)$ (robust mean and standard deviation, Leys et al. 2013;
  recommended over mean ± k·std for PV arrays by Zhao et al. 2013). The same limits are used for validation and testing.
- Prediction error (residual) as the anomaly signal: Malhotra et al. 2015, Hundman et al. 2018; residual of a PV
  system against a physical model with a k·σ control limit: Garoudja et al. 2017, Harrou et al. 2018.
- Readings with irradiation below 0.2 kW/m² (dawn and dusk) are **not evaluated** (flag 0, `EVALUATED = 0`):
  the expected power there is a few kW, so every model's relative error is large (RdTools filter, 200 W/m²).

**Why not the previous rules?** The previous rules looked at each power line on its own (change from the previous
reading, or the usual power at this time of day). The LSTM line only follows the weather, so it can never show an
outage, and the actual line is flagged for clouds as well as for faults. Notebook 4 measures this.

## Inputs

In [1]:
import os, shutil, zipfile
import numpy as np
import pandas as pd

def find_root():
    for d in ['.', '..', '../..']:
        if os.path.exists(os.path.join(d, 'data', 'raw', 'Plant_1_Generation_Data.csv')):
            return os.path.abspath(d)
    return os.path.abspath('.')

ROOT = find_root()
OUTPUTS = os.path.join(ROOT, 'outputs')

def get_step(name):
    folder = os.path.join(OUTPUTS, name)
    if not os.path.isdir(folder):
        zip_path = os.path.join(OUTPUTS, name + '.zip') if os.path.exists(os.path.join(OUTPUTS, name + '.zip')) else name + '.zip'
        if not os.path.exists(zip_path):
            from google.colab import files
            print(f'Upload {name}.zip'); uploaded = files.upload(); zip_path = list(uploaded)[0]
        shutil.unpack_archive(zip_path, folder, 'zip')
    return folder

STEP2_DIR = get_step('step2_outputs')
OUT_DIR = os.path.join(OUTPUTS, 'step3_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

settings = dict(line.strip().split(' = ') for line in open(os.path.join(STEP2_DIR, 'settings.txt')) if ' = ' in line)
IRR_MIN = float(settings['IRR_MIN'])
K_VALUES = [1.0, 1.5, 2.0, 2.5, 3.0]
SPLITS = ['train', 'val', 'test']
print('IRR_MIN =', IRR_MIN, '| k values:', K_VALUES)

IRR_MIN = 0.2 | k values: [1.0, 1.5, 2.0, 2.5, 3.0]


## 1. The anomaly function

In [2]:
def robust_mu_sigma(x):
    '''Median and 1.4826 * MAD (= mean and std for normal data, not pulled by faults).'''
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    med = np.median(x)
    return med, 1.4826 * np.median(np.abs(x - med))

def fit_limits(train_df, expected_col):
    '''mu and sigma of r = actual - expected, per inverter, from the TRAINING readings that are evaluated.'''
    t = train_df[train_df['EVALUATED'] == 1]
    r = t['ACTUAL_AC_POWER_kW'] - t[expected_col]
    return pd.DataFrame({inv: robust_mu_sigma(g) for inv, g in r.groupby(t['SOURCE_KEY'])}, index=['mu', 'sigma']).T

def detect_anomalies(df, expected_col, limits, k):
    '''1 if actual - expected is outside mu ± k*sigma of this inverter, else 0. Not-evaluated readings get 0.'''
    r = df['ACTUAL_AC_POWER_kW'] - df[expected_col]
    mu = df['SOURCE_KEY'].map(limits['mu'])
    sigma = df['SOURCE_KEY'].map(limits['sigma'])
    flag = ((r < mu - k * sigma) | (r > mu + k * sigma)) & (df['EVALUATED'] == 1)
    return flag.astype(int).values

## 2. Apply the rule to each line, for every k

- `actual_anomaly_k{k}`: actual vs physical expected power (ground truth)
- `pred_anomaly_k{k}`: actual vs LSTM trained on healthy readings (main model)
- `pred_all_anomaly_k{k}`: actual vs LSTM trained on all readings (experiment, Notebook 4)

In [3]:
data = {}
for s in SPLITS:
    df = pd.read_csv(os.path.join(STEP2_DIR, f'prediction_{s}.csv'), parse_dates=['DATE_TIME'])
    df = df.sort_values(['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)
    df['EVALUATED'] = (df['IRRADIATION'] >= IRR_MIN).astype(int)
    data[s] = df

LINES = {'actual': 'EXPECTED_PHYS_kW', 'pred': 'PREDICTED_AC_POWER_kW', 'pred_all': 'PREDICTED_ALL_DATA_kW'}
limits = {name: fit_limits(data['train'], col) for name, col in LINES.items()}     # training days only

for s in SPLITS:
    df = data[s]
    for name, col in LINES.items():
        df[f'RESID_{name.upper()}_kW'] = df['ACTUAL_AC_POWER_kW'] - df[col]
        for k in K_VALUES:
            df[f'{name}_anomaly_k{k}'] = detect_anomalies(df, col, limits[name], k)
    print(f'\n{s.upper()}: {len(df):,} readings, evaluated {df.EVALUATED.sum():,}')
    for k in K_VALUES:
        print(f'  k={k}: Actual {df[f"actual_anomaly_k{k}"].sum():5d} | Predicted {df[f"pred_anomaly_k{k}"].sum():5d} '
              f'| Predicted (all-data LSTM) {df[f"pred_all_anomaly_k{k}"].sum():5d}')


TRAIN: 36,491 readings, evaluated 24,170
  k=1.0: Actual  8317 | Predicted  9207 | Predicted (all-data LSTM)  9600
  k=1.5: Actual  5630 | Predicted  6731 | Predicted (all-data LSTM)  7090
  k=2.0: Actual  4489 | Predicted  5415 | Predicted (all-data LSTM)  5173
  k=2.5: Actual  3876 | Predicted  4583 | Predicted (all-data LSTM)  3505
  k=3.0: Actual  3481 | Predicted  4076 | Predicted (all-data LSTM)  2186

VAL: 11,968 readings, evaluated 7,458
  k=1.0: Actual  2258 | Predicted  2700 | Predicted (all-data LSTM)  2120
  k=1.5: Actual  1525 | Predicted  1921 | Predicted (all-data LSTM)  1489
  k=2.0: Actual  1165 | Predicted  1479 | Predicted (all-data LSTM)  1093
  k=2.5: Actual   935 | Predicted  1218 | Predicted (all-data LSTM)   724
  k=3.0: Actual   827 | Predicted  1070 | Predicted (all-data LSTM)   429

TEST: 10,758 readings, evaluated 6,930
  k=1.0: Actual  1976 | Predicted  2496 | Predicted (all-data LSTM)  1904
  k=1.5: Actual  1331 | Predicted  1610 | Predicted (all-data LST

In [4]:
# The limits (training only), per inverter and line
lim = pd.concat({name: l for name, l in limits.items()}, axis=1)
lim.columns = [f'{a}_{b}' for a, b in lim.columns]
lim.index.name = 'SOURCE_KEY'
lim.to_csv(os.path.join(OUT_DIR, 'limits_mu_sigma.csv'))
print('sigma (kW) over the 44 inverters:')
lim[[c for c in lim.columns if c.endswith('sigma')]].describe().round(1)

sigma (kW) over the 44 inverters:


,actual_sigma,pred_sigma,pred_all_sigma
count,44.0,44.0,44.0
mean,66.5,42.9,124.7
std,22.4,14.7,80.8
min,40.9,30.8,39.9
25%,50.2,34.2,51.0
50%,59.5,41.1,107.2
75%,77.7,45.8,185.8
max,137.8,120.8,316.7


## 3. Save and zip

In [5]:
for s, df in data.items():
    df.to_csv(os.path.join(OUT_DIR, f'{s}_with_anomalies.csv'), index=False)
zip_path = shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Saved to', zip_path)

['limits_mu_sigma.csv', 'test_with_anomalies.csv', 'train_with_anomalies.csv', 'val_with_anomalies.csv']
Saved to /home/user/SUNSHIELD/outputs/step3_outputs.zip
